In [ ]:
from pathlib import Path
import shutil
import yaml
from tqdm import tqdm

In [ ]:
# Original dataset
ANTI_UAV_ROOT = Path(
    r"D:\master\Master_Drone_Detection\datasets\AntiUAv-RGPT DataSet"
)


# New YOLO dataset
YOLO_ROOT = Path(
    r"D:\master\Master_Drone_Detection\datasets\AntiUAV_YOLO"
)


splits = ["train", "val", "test"]

In [ ]:
for split in splits:

    (YOLO_ROOT / "images" / split).mkdir(
        parents=True,
        exist_ok=True
    )

    (YOLO_ROOT / "labels" / split).mkdir(
        parents=True,
        exist_ok=True
    )

In [ ]:
def convert_sequence(sequence_path, output_img, output_lbl):

    visible_dir = sequence_path / "visible"
    label_dir = sequence_path / "labels"


    images = sorted(
        visible_dir.glob("*.jpg")
    )


    for img_path in images:

        label_path = label_dir / (
            img_path.stem + ".txt"
        )


        # copy image

        shutil.copy(
            img_path,
            output_img / img_path.name
        )


        yolo_label = output_lbl / (
            img_path.stem + ".txt"
        )


        # if no annotation
        if not label_path.exists():

            continue


        new_lines=[]


        with open(label_path,"r") as f:

            lines=f.readlines()


        for line in lines:

            values=line.strip().split()


            if len(values)==4:

                x,y,w,h=values

            else:
                continue


            # class UAV = 0

            new_lines.append(
                f"0 {x} {y} {w} {h}"
            )


        if new_lines:

            with open(yolo_label,"w") as f:

                f.write(
                    "\n".join(new_lines)
                )

In [ ]:
for split in splits:

    print(
        f"Processing {split}"
    )


    split_path = ANTI_UAV_ROOT / split


    output_img = YOLO_ROOT / "images" / split

    output_lbl = YOLO_ROOT / "labels" / split


    sequences = list(
        split_path.iterdir()
    )


    for seq in tqdm(sequences):

        if seq.is_dir():

            convert_sequence(
                seq,
                output_img,
                output_lbl
            )


print("Conversion completed")

In [ ]:
for split in splits:

    images = list(
        (YOLO_ROOT/"images"/split).glob("*.jpg")
    )

    labels = list(
        (YOLO_ROOT/"labels"/split).glob("*.txt")
    )


    print(
        split,
        "Images:",
        len(images),
        "Labels:",
        len(labels)
    )

In [ ]:
data_yaml = {

    "path": str(YOLO_ROOT),

    "train": "images/train",

    "val": "images/val",

    "test": "images/test",

    "names": {
        0:"UAV"
    }

}


with open(
    YOLO_ROOT/"data.yaml",
    "w"
) as f:

    yaml.dump(
        data_yaml,
        f,
        sort_keys=False
    )